[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/TP_dimension_effective.ipynb)

# Dimension effective du paysage d'énergie

Autour d'une image $x$, à l'échelle $\sqrt t$, la dimension effective $d_{\text{eff}}(x,t) = \frac1t\mathbb E_y\big[\|x - \mathbb E[x\mid y]\|^2 \mid x\big]$ compte les directions dans lesquelles le débruiteur optimal laisse passer le bruit (équation (11)). Elle se lit aussi sur l'énergie effective $\mathbb E_y[U(y,t)\mid x]$ (équation (12)). On reprend la figure 5 de l'article, on compare les deux définitions, un modèle gaussien, et l'on reconstruit $\log p(x)$ à partir des erreurs de débruitage (équation (16)).

**Matériel** : tourne sur un ordinateur personnel (CPU) en quelques minutes ; sur Colab, exécuter d'abord la cellule d'installation.

**Profils** : profil réduit (par défaut), 6 des 15 images de référence des auteurs, 25 niveaux de bruit et 4 réalisations du bruit (environ 2 minutes sur CPU) ; profil GPU, les 15 images, 50 niveaux et 32 réalisations (les auteurs en prennent 100 et 1000). Notebook des auteurs correspondant : `auteurs/dimensionality_colab.ipynb`.

Le profil réduit est utilisé par défaut, même avec un GPU (le calcul se fait alors sur le GPU) ; pour les réglages des auteurs, remplacer `if False:` par `if True:` dans la cellule de réglages (profil GPU, non vérifié).

**Données** : la cellule « Données », après l'installation, télécharge les images (417 Mo) ; elle ne fait rien par défaut : remplacer `if False:` par `if True:` au premier lancement.

**Reprise** : les calculs longs sont mis en cache dans `resultats/cache/` ; une exécution interrompue reprend là où elle s'était arrêtée. Supprimer ce dossier pour tout recalculer.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py',
            '/models/finalclean_imagenet64_grayscale_lr0.0005_1Msteps10decays/',
            '/outputs/dimensionalities_ref.pt',
            '/outputs/dimensionalities_avg.pt']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, "tp"))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, "tp"))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

In [ ]:
# Données (une fois par machine ; sur Colab, à chaque nouvelle session) : images de validation d'ImageNet64
# (50 000 images), téléchargées depuis un miroir HuggingFace (417 Mo), décodées (quelques minutes) et enregistrées dans
# data/ à la racine du dépôt. Rien n'est téléchargé par défaut : remplacer False par True ci-dessous.
if False:
    import tp_outils
    tp_outils.preparer_imagenet64_val()

## 0. Réglages et images

On travaille sur les images de référence des auteurs (figure 3 de l'article : log-probabilités régulièrement
espacées sur le jeu de test), avec leur grille de niveaux de bruit ($10^{-12}$ à $10^3$, sous-échantillonnée), pour
comparer nos courbes aux leurs (partie 5).

In [ ]:
%matplotlib inline
import math, os
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_dimension_effective")

# Profil : réduit par défaut, que le calcul se fasse sur CPU ou sur GPU (T.DEVICE suit le matériel connecté).
# Réglages des auteurs (profil GPU, non vérifié, GPU indispensable) : remplacer False par True ci-dessous.
GPU = False
if False:
    GPU = torch.cuda.is_available()
CHOIX = list(range(15)) if GPU else [0, 3, 6, 8, 11, 14]   # parmi les 15 images de référence des auteurs
PAS_T = 2 if GPU else 4                # un niveau de bruit sur PAS_T de la grille des auteurs (100 niveaux)
N_BRUITS = 32 if GPU else 4            # réalisations du bruit par image et par niveau
N_MOYENNE = 512 if GPU else 32         # images pour la courbe moyenne
GRAINE = 0
print("calcul sur", T.DEVICE)
m = T.charger_modele("gris")
d = m.d

ts_a = torch.logspace(-12, 3, 100)     # grille des auteurs
ts = ts_a[::PAS_T]
Xref = T.charger_imagenet64_val(indices=[T.IDX_REF_AUTEURS[k] for k in CHOIX])
lp_ref = m.log_probabilites(Xref)
N_REF = len(CHOIX)
couleurs = plt.cm.plasma(np.array(CHOIX) / 14)
T.afficher_images(Xref, ["%.1f dB" % v for v in lp_ref], ncol=N_REF, taille=1.4, nom="images")

## 1. Un modèle de référence : la gaussienne

Pour $x \sim \mathcal N(\mu, \Sigma)$, le débruiteur optimal est linéaire (filtre de Wiener) et son erreur ne
dépend pas de $x$ : $d_{\text{eff}}(t) = \sum_i \frac{\lambda_i}{\lambda_i + t}$, où les $\lambda_i$ sont les
valeurs propres de $\Sigma$. On utilise la covariance de 10 000 images de validation.

In [ ]:
Xcov = T.charger_imagenet64_val(10_000, debut=40_000).flatten(1)
cen = Xcov - Xcov.mean(0)
lam = torch.linalg.eigvalsh(cen.T @ cen / len(cen)).clamp_min(0).double()
d_gauss = torch.stack([(lam / (lam + t)).sum() for t in ts.double()])
for t in [1e-6, 1e-4, 1e-2, 1.0]:
    print("t = %.0e : d_eff gaussien = %.0f" % (t, (lam / (lam + t)).sum()))
print("nombre de valeurs propres > 1/256^2 : %d" % (lam > 1 / 256 ** 2).sum())

## 2. Énergie et dimension effectives (figure 5)

Pour chaque image et chaque $t$ : $y = x + \sqrt t\,z$, énergie $U_\theta(y, t)$ et erreur
$\|x - \hat x(y)\|^2/t$ avec $\hat x = y - t\nabla_yU_\theta$, moyennées sur les réalisations du bruit
(`T.energie_effective`). Courbe moyenne sur `N_MOYENNE` autres images.

In [ ]:
import time; debut = time.time()
E, D = T.energie_effective(m, Xref, ts, n_bruits=N_BRUITS, graine=GRAINE, cache="references")
Xmoy = T.charger_imagenet64_val(N_MOYENNE, debut=45_000)
Em, Dm = T.energie_effective(m, Xmoy, ts, n_bruits=1, graine=GRAINE, cache="moyenne")
Em, Dm = Em.mean(0), Dm.mean(0)
print("calcul en %.0f s" % (time.time() - debut))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for k in range(N_REF):
    axes[0].semilogx(ts, T.nats_vers_dB(E[k], d), color=couleurs[k])
    axes[1].semilogx(ts, D[k], color=couleurs[k])
axes[0].semilogx(ts, T.nats_vers_dB(Em, d), "k--", label="moyenne")
axes[1].semilogx(ts, Dm, "k--", label="moyenne")
axes[1].semilogx(ts, d_gauss, color="gray", ls=":", label="modèle gaussien")
for ax in axes:
    ax.axvline(T.T_MIN, color="gray", lw=0.8); ax.set_xlabel("variance du bruit t"); ax.legend(fontsize=8)
axes[0].set_ylabel("E[log p(y, t) | x] (dB/dim)"); axes[0].set_ylim(-20, 45)
axes[1].axhline(d, color="gray", lw=0.8); axes[1].set_ylabel("d_eff(x, t)"); axes[1].set_ylim(0, 4300)
T.montrer("energie_dimension")
for t in [1e-8, 1e-5, 1e-3, 1e-1]:
    j = int(torch.argmin((ts - t).abs()))
    print("t = %.0e : d_eff de l'image la moins probable %.0f, la plus probable %.0f, moyenne %.0f, gaussien %.0f"
          % (ts[j], D[0, j], D[-1, j], Dm[j], d_gauss[j]))

## 3. Deux définitions de la dimension effective

Équation (12) : $d_{\text{eff}}(x,t) = d - \partial_{\frac12\log t}\,\mathbb E_y[U(y,t)\mid x] = d - 2t\,\partial_t
\mathbb E_y[U(y,t)\mid x]$, estimée par différences finies sur la grille de $t$. Elle ne coïncide avec
l'équation (11) que si le modèle vérifie l'équation de diffusion.

In [ ]:
lt = torch.log(ts)
deff12 = d - 2 * (E[:, 1:] - E[:, :-1]) / (lt[1:] - lt[:-1])
tm = torch.exp((lt[1:] + lt[:-1]) / 2)
fig, ax = plt.subplots(figsize=(6, 3.8))
for k in [0, N_REF // 2, N_REF - 1]:
    ax.semilogx(ts, D[k], color=couleurs[k], label="(11), log p = %.1f" % lp_ref[k])
    ax.semilogx(tm, deff12[k], color=couleurs[k], ls="--", marker=".", label="(12)")
ax.set_xlabel("t"); ax.set_ylabel("d_eff"); ax.set_ylim(-500, 4500); ax.legend(fontsize=7, ncol=2)
T.montrer("deux_definitions")

## 4. Retrouver log p(x) à partir des erreurs de débruitage

Équation (16) : $-\log p_{\text{MSE}}(x) = \frac d2 \log(2\pi e\, t_{\max}) - \int_0^{t_{\max}}
\big(d - d_{\text{eff}}(x,t)\big)\frac{dt}{2t}$, majorant de $-\log p(x)$ quand $\hat x$ n'est pas le
débruiteur optimal. On l'évalue par la méthode des trapèzes en $\log t$ (l'intégrande est négligeable sous
$10^{-10}$) et on la compare à l'évaluation directe $U_\theta(x, 0)$.

In [ ]:
integrale = torch.trapezoid((d - D) / 2, lt, dim=1)
nll_mse = d / 2 * math.log(2 * math.pi * math.e * ts[-1]) - integrale
nll_direct = m.energie(Xref)
print("%10s %18s %18s" % ("log p direct", "log p par (16)", "écart (dB/dim)"))
for a, b in zip(T.nats_vers_dB(nll_direct, d), T.nats_vers_dB(nll_mse, d)):
    print("%10.2f %18.2f %18.2f" % (a, b, b - a))

## 5. Les courbes des auteurs

Les auteurs fournissent leurs courbes (`outputs/dimensionalities_*.pt`) pour les mêmes 15 images : 100 niveaux de
bruit, 1 000 réalisations du bruit, moyenne sur les 50 000 images de test. On superpose nos estimations (points).

In [ ]:
E_a, D_a = torch.load(os.path.join(T.RACINE, "outputs", "dimensionalities_ref.pt"), map_location="cpu")
Em_a, Dm_a = torch.load(os.path.join(T.RACINE, "outputs", "dimensionalities_avg.pt"), map_location="cpu")
ts_a = torch.logspace(-12, 3, 100)
coul_a = plt.cm.plasma(np.linspace(0, 1, 15))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for k in range(15):
    axes[0].semilogx(ts_a, -E_a[k], color=coul_a[k]); axes[1].semilogx(ts_a, D_a[k], color=coul_a[k])
for i, k in enumerate(CHOIX):
    axes[0].semilogx(ts, T.nats_vers_dB(E[i], d), "o", ms=3, color=coul_a[k])
    axes[1].semilogx(ts, D[i], "o", ms=3, color=coul_a[k])
axes[0].semilogx(ts_a, -Em_a, "k--"); axes[1].semilogx(ts_a, Dm_a, "k--")
axes[0].set_ylabel("E[log p(y,t) | x] (dB/dim)"); axes[0].set_ylim(-20, 40)
axes[1].set_ylabel("d_eff"); axes[1].set_ylim(0, 4300)
for ax in axes:
    ax.axvline(1e-9, color="gray", lw=0.8); ax.set_xlabel("t")
T.montrer("courbes_auteurs")
j = int(torch.argmin((ts_a - 1e-4).abs()))
print("auteurs, t = 1e-4 : d_eff de %.0f à %.0f selon l'image, moyenne %.0f" % (D_a[:, j].min(), D_a[:, j].max(), Dm_a[j]))
ecart_d = (D - D_a[CHOIX][:, ::PAS_T]).abs()
ecart_e = (T.nats_vers_dB(E, d) + E_a[CHOIX][:, ::PAS_T]).abs()
print("écart à leurs courbes (mêmes images) : d_eff médian %.0f, max %.0f ; énergie effective médiane %.3f, max %.3f dB/dim"
      % (ecart_d.median(), ecart_d.max(), ecart_e.median(), ecart_e.max()))